<a href="https://colab.research.google.com/github/ai1108/budugudao-elderly-care-platform/blob/main/gis_data_pipeline/gis_data_pipeline_elderly_care.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 《不孤島》高齡生活風險評估 —— GIS 空間資料蒐集與指標建構

本 Notebook 負責專題中「GIS 空間資料蒐集與處理」子任務，目標：

1. 蒐集臺北市 / 新北市之行政區、村里界線、交通、公園綠地、生活機能、
   高齡友善設施、無障礙設施、醫療與長照機構、道路網路等原始 GIS 資料
2. 計算各項空間指標（最近距離、路網距離、Buffer 可近性、服務範圍涵蓋率等）
3. 依「村里」與「行政區」彙整，輸出為 CSV，供後續風險評估模型使用

> **執行前提**：本機需能連上網路（下載 OSM 資料 / 政府開放資料 API）。
> 建議在 Google Colab 或本機 Jupyter 執行，並先安裝下方套件。

> **重要提醒**：部分政府開放資料（村里界線 shapefile、公園圖資、無障礙設施等）
> 目前多以「手動下載檔案」形式提供，而非穩定 API。本 Notebook 已將這些資料源
> 設計為「讀取本地檔案路徑」，請依照 `config` 區塊中的說明，先至對應網站下載
> 後放入 `data/raw/` 資料夾。OSM 相關資料（路網、生活機能 POI）則可直接用
> OSMnx 自動下載，不需手動處理。


## 給接手組員的交接說明

**你需要的金鑰**：TDX 平台的 **API 金鑰（Client ID + Client Secret）**，
不是 MQTT 金鑰。API 金鑰用來查詢公車站牌、捷運車站等「靜態站點資料」；
MQTT 金鑰是給即時動態資料用的，這份研究用不到。

**最快上手方式（不需要先手動下載任何村里界線檔案）**：

1. 安裝套件（見「0. 環境設定」）
2. 直接跳到本 Notebook 最下方 **「快速示範：單一行政區」** 區塊
3. 在 `TDX_CLIENT_ID` / `TDX_CLIENT_SECRET` 填入你的金鑰
4. 依序執行該區塊所有 cell，即可在 `outputs/` 資料夾看到一份
   單一行政區（預設「大安區」）的示範 CSV
5. 若要換成其他行政區，只要把 `DEMO_DISTRICT_EN` / `DEMO_DISTRICT_ZH`
   改成想要的區名即可（見該區塊說明）

> 若之後要做完整臺北市＋新北市、村里層級的正式分析，才需要回頭執行
> 第 1～9 節的完整流程（那部分需要額外下載村里界線 shapefile）。
> 快速示範不需要這個檔案，它直接用 OSMnx 抓行政區邊界，範圍縮小到
> 一個區，執行速度快很多，適合先跑出一份能看的成果。


## 0. 環境設定與套件安裝

In [ ]:
# 首次執行請取消註解安裝（Colab 建議直接執行）
!pip install geopandas osmnx networkx shapely pyproj scikit-learn pandas numpy requests tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.7/104.7 kB 5.3 MB/s eta 0:00:00


In [ ]:
import os
import json
import time
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, Polygon, LineString
from shapely.ops import unary_union
from sklearn.neighbors import BallTree
import networkx as nx
import osmnx as ox

# ---- 全域座標系統設定 ----
# WGS84（大部分政府資料/OSM 原始座標）
CRS_WGS84 = "EPSG:4326"
# TWD97 119分帶（公尺為單位，用於距離、面積、buffer 運算）
CRS_TWD97 = "EPSG:3826"

pd.set_option("display.max_columns", 50)


In [ ]:
# ---- 路徑與研究範圍設定 ----
RAW_DIR = "data/raw"
PROCESSED_DIR = "data/processed"
OUTPUT_DIR = "outputs"

for d in [RAW_DIR, PROCESSED_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

# 研究範圍：臺北市、新北市
STUDY_CITIES = ["臺北市", "新北市"]

# 村里界線 shapefile / geojson 路徑（請先至下方資料來源下載後放入 RAW_DIR）
# 來源：內政部社會經濟資料服務平台 https://segis.moi.gov.tw/
#       或 TGOS 圖資雲服務平台 https://www.tgos.tw/
VILLAGE_BOUNDARY_PATH = os.path.join(RAW_DIR, "village_boundary.shp")   # 村里界
DISTRICT_BOUNDARY_PATH = os.path.join(RAW_DIR, "district_boundary.shp") # 行政區界（可由村里界 dissolve 取得）


## 1. 行政區域界線（村里 / 行政區）

政府開放資料多提供 shapefile（.shp + .dbf + .shx + .prj）壓縮檔，下載後解壓至
`data/raw/`。若檔案為 TWD97 座標但未正確標示 CRS，需手動指定。


In [ ]:
def load_village_boundary(path=VILLAGE_BOUNDARY_PATH, cities=STUDY_CITIES):
    """讀取村里界線圖資，篩選研究範圍縣市，統一轉換為 TWD97。"""
    gdf = gpd.read_file(path)

    # 若讀入後沒有 CRS 資訊，依資料來源實際情況指定（常見為 TWD97 或 WGS84）
    if gdf.crs is None:
        gdf = gdf.set_crs(CRS_TWD97)
    gdf = gdf.to_crs(CRS_TWD97)

    # 欄位名稱依實際下載資料調整，常見欄位如 COUNTYNAME / TOWNNAME / VILLNAME
    rename_map = {
        "COUNTYNAME": "county",
        "TOWNNAME": "district",
        "VILLNAME": "village",
        "VILLCODE": "village_code",
    }
    gdf = gdf.rename(columns={k: v for k, v in rename_map.items() if k in gdf.columns})

    if "county" in gdf.columns:
        gdf = gdf[gdf["county"].isin(cities)].reset_index(drop=True)

    gdf["village_area_km2"] = gdf.geometry.area / 1_000_000
    return gdf

# village_gdf = load_village_boundary()
# village_gdf.head()


In [ ]:
def build_district_boundary(village_gdf):
    """由村里界 dissolve 產生行政區界線。"""
    district_gdf = village_gdf.dissolve(by=["county", "district"], as_index=False)
    district_gdf["district_area_km2"] = district_gdf.geometry.area / 1_000_000
    return district_gdf[["county", "district", "district_area_km2", "geometry"]]

# district_gdf = build_district_boundary(village_gdf)


In [ ]:
def get_village_centroids(village_gdf):
    """取得每個村里的幾何中心點，作為後續可近性/距離運算的代表點。
    若有人口網格資料，建議改用「人口重心」取代幾何中心點，精確度更高。"""
    pts = village_gdf.copy()
    pts["geometry"] = pts.geometry.centroid
    return pts[["county", "district", "village", "village_code", "geometry"]]

# village_centroids = get_village_centroids(village_gdf)


## 2. 道路網路資料（OSMnx）

直接透過 OSMnx 抓取臺北市／新北市的步行與道路路網，作為路網距離、
旅行時間與服務範圍分析的基礎圖。


In [ ]:
def download_road_network(place_names, network_type="walk"):
    """network_type: 'walk'（人行可近性用）或 'drive'（就醫可近性用）"""
    graphs = []
    for place in place_names:
        g = ox.graph_from_place(place, network_type=network_type)
        graphs.append(g)
    G = graphs[0]
    for g in graphs[1:]:
        G = nx.compose(G, g)
    # 加上邊的旅行時間（步行 4.5km/h，開車依道路速限，OSMnx 可自動補推估速限）
    G = ox.add_edge_speeds(G)
    G = ox.add_edge_travel_times(G)
    return G

PLACE_NAMES = ["Taipei, Taiwan", "New Taipei, Taiwan"]

# G_walk = download_road_network(PLACE_NAMES, network_type="walk")
# G_drive = download_road_network(PLACE_NAMES, network_type="drive")


## 3. 各類設施 / POI 資料蒐集

### 3.1 OSM 生活機能 POI（超商、市場、藥局、公園等）——可直接自動化下載


In [ ]:
# OSM tag 對照表：依需求擴充
OSM_TAGS = {
    "convenience_store": {"shop": "convenience"},
    "supermarket": {"shop": "supermarket"},
    "market": {"amenity": "marketplace"},
    "pharmacy": {"amenity": "pharmacy"},
    "clinic": {"amenity": "clinic"},
    "hospital": {"amenity": "hospital"},
    "park": {"leisure": "park"},
    "community_center": {"amenity": "community_centre"},
    "bus_stop": {"highway": "bus_stop"},
    "subway_station": {"railway": "station", "station": "subway"},
}

def download_osm_pois(place_names, tags):
    """下載指定 tags 的 POI，回傳 GeoDataFrame（WGS84）。"""
    frames = []
    for place in place_names:
        try:
            gdf = ox.features_from_place(place, tags)
            frames.append(gdf)
        except Exception as e:
            print(f"{place} - {tags} 下載失敗：{e}")
    if not frames:
        return gpd.GeoDataFrame(columns=["geometry"], crs=CRS_WGS84)
    result = pd.concat(frames, ignore_index=True)
    result = gpd.GeoDataFrame(result, crs=CRS_WGS84)
    # 面/線資料統一轉為代表點，方便後續距離運算
    result["geometry"] = result.geometry.apply(
        lambda geom: geom.centroid if geom.geom_type != "Point" else geom
    )
    return result.to_crs(CRS_TWD97)

# poi_dict = {}
# for name, tags in OSM_TAGS.items():
#     poi_dict[name] = download_osm_pois(PLACE_NAMES, tags)
#     print(name, len(poi_dict[name]))


### 3.2 政府開放資料 API（已驗證之實際資料來源）

以下網址與欄位名稱皆已實際查證（2026 年 8 月），可直接於程式中呼叫，
**不需要另外手動下載檔案**：

| 資料 | 來源 | 是否含經緯度 |
|---|---|---|
| 長照ABC據點（巷弄長照站/複合型/社區整合型） | 衛福部，`data.gov.tw/dataset/88270` | ✅ 有「經度、緯度」欄位 |
| 台北捷運車站 | TDX `/v2/Rail/Metro/Station/TRTC` | ✅ 有 StationPosition |
| 公車站牌（台北市/新北市） | TDX `/v2/Bus/Stop/City/{City}` | ✅ 有 StopPosition |
| 健保特約醫事機構（醫院/診所/藥局） | 健保署，`data.gov.tw/dataset/39280~39284` | ❌ 只有地址，需自行地理編碼 |
| 新北市各類設施（公園、無障礙停車格等） | 新北市資料開放平臺（您提供的 `data.ntpc.gov.tw/openapi`） | 視資料集而定 |
| 台北市無障礙友善店家 | `data.gov.tw/dataset/147918` | ✅ 有「經度、緯度」欄位 |

**TDX 申請說明**：至 https://tdx.transportdata.tw/ 註冊會員（學生可用學校信箱申請「學研單位」），
在「會員中心 > 資料服務 > API 金鑰」取得 Client ID / Secret，即可呼叫下方函式。


In [ ]:
def get_tdx_token(client_id, client_secret):
    """取得 TDX API 存取權杖。"""
    auth_url = "https://tdx.transportdata.tw/auth/realms/TDXConnect/protocol/openid-connect/token"
    resp = requests.post(
        auth_url,
        data={
            "grant_type": "client_credentials",
            "client_id": client_id,
            "client_secret": client_secret,
        },
    )
    resp.raise_for_status()
    return resp.json()["access_token"]


def fetch_tdx_bus_stops(token, city="Taipei"):
    """取得指定縣市公車站牌資料。city 可為 'Taipei' 或 'NewTaipei'。"""
    api_url = f"https://tdx.transportdata.tw/api/basic/v2/Bus/Stop/City/{city}?%24format=JSON"
    resp = requests.get(api_url, headers={"authorization": f"Bearer {token}"})
    resp.raise_for_status()
    data = resp.json()

    records = []
    for stop in data:
        pos = stop.get("StopPosition", {})
        records.append({
            "stop_id": stop.get("StopID"),
            "stop_name": stop.get("StopName", {}).get("Zh_tw"),
            "lon": pos.get("PositionLon"),
            "lat": pos.get("PositionLat"),
        })
    df = pd.DataFrame(records).dropna(subset=["lon", "lat"])
    geometry = [Point(xy) for xy in zip(df["lon"], df["lat"])]
    gdf = gpd.GeoDataFrame(df, geometry=geometry, crs=CRS_WGS84)
    return gdf.to_crs(CRS_TWD97)


def fetch_tdx_metro_stations(token, rail_system="TRTC"):
    """取得捷運車站資料。rail_system: 'TRTC'（台北捷運）。
    新北捷運（淡海輕軌）另有代碼，請參考 TDX_Railway 對照表。"""
    api_url = f"https://tdx.transportdata.tw/api/basic/v2/Rail/Metro/Station/{rail_system}?%24format=JSON"
    resp = requests.get(api_url, headers={"authorization": f"Bearer {token}"})
    resp.raise_for_status()
    data = resp.json()

    records = []
    for st in data:
        pos = st.get("StationPosition", {})
        records.append({
            "station_id": st.get("StationID"),
            "station_name": st.get("StationName", {}).get("Zh_tw"),
            "lon": pos.get("PositionLon"),
            "lat": pos.get("PositionLat"),
        })
    df = pd.DataFrame(records).dropna(subset=["lon", "lat"])
    geometry = [Point(xy) for xy in zip(df["lon"], df["lat"])]
    gdf = gpd.GeoDataFrame(df, geometry=geometry, crs=CRS_WGS84)
    return gdf.to_crs(CRS_TWD97)

# 範例（填入自己申請的金鑰後即可執行）：
# token = get_tdx_token(client_id="YOUR_ID", client_secret="YOUR_SECRET")
# bus_stops_tp = fetch_tdx_bus_stops(token, city="Taipei")
# bus_stops_ntp = fetch_tdx_bus_stops(token, city="NewTaipei")
# metro_stations = fetch_tdx_metro_stations(token, rail_system="TRTC")
# transit_gdf = pd.concat([bus_stops_tp, bus_stops_ntp, metro_stations], ignore_index=True)
# transit_gdf = gpd.GeoDataFrame(transit_gdf, crs=CRS_TWD97)


In [ ]:
def fetch_ltc_abc_facilities(cities=("台北市", "新北市")):
    """取得長照ABC據點資料（衛福部，每日更新，已含經緯度，不需金鑰）。
    資料集頁：https://data.gov.tw/dataset/88270"""
    # 資料集頁面之 CSV 下載網址（若機關更新資源網址，請至上方連結重新確認）
    url = "https://data.gov.tw/api/v2/rest/dataset/88270"
    meta = requests.get(url).json()
    csv_url = meta["result"]["distribution"][0]["resourceDownloadUrl"]

    df = pd.read_csv(csv_url)
    # 欄位名稱依機關命名，常見為「經度」「緯度」「特約縣市」
    df = df[df["特約縣市"].isin(cities)].copy()
    df = df.dropna(subset=["經度", "緯度"])

    geometry = [Point(xy) for xy in zip(df["經度"], df["緯度"])]
    gdf = gpd.GeoDataFrame(df, geometry=geometry, crs=CRS_WGS84)
    return gdf.to_crs(CRS_TWD97)

# ltc_facilities = fetch_ltc_abc_facilities()
# ltc_facilities.head()


### 3.3 需要地理編碼的資料（健保特約醫事機構）

健保署提供的醫療院所清冊**只有地址、沒有經緯度**，需先做地理編碼（address → lat/lon）。
兩種常見做法：

1. **TGOS 全國門牌地址定位服務**（內政部）：精度最高，但需申請會員資格
   （限政府機關、法人、學術單位、業界），見 https://www.tgos.tw/tgos/Addr
2. **Nominatim / OpenStreetMap 地理編碼**（`geopy` 套件）：免申請、免費，
   但地址需相對完整，且有請求頻率限制（建議每筆間隔 1 秒以上）

以下示範用 `geopy` 的免費方案，資料量大時建議申請 TGOS 以提升精度與速度。


In [ ]:
# !pip install geopy --break-system-packages
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

def fetch_nhi_medical_institutions(resource_id, city_names=("台北市", "新北市")):
    """取得健保特約醫事機構清冊（僅含地址，不含座標）。
    resource_id 範例：
      - 診所：A21030000I-D21004-009
      - 醫學中心/區域醫院/地區醫院：至 data.gov.tw 對應資料集頁面查詢 resource ID
    """
    csv_url = f"https://info.nhi.gov.tw/api/iode0000s01/Dataset?rId={resource_id}"
    df = pd.read_csv(csv_url, encoding="utf-8")
    # 欄位包含：醫事機構代碼、醫事機構名稱、醫事機構種類、電話、地址、縣市別代碼 等
    if "地址" in df.columns:
        df = df[df["地址"].str.contains("|".join(city_names), na=False)].copy()
    return df


def geocode_addresses(df, address_col="地址", delay_sec=1.1):
    """用 Nominatim 免費地理編碼服務，將地址轉為經緯度，回傳 GeoDataFrame。
    大量資料建議改申請 TGOS 服務以提升效率與精確度。"""
    geolocator = Nominatim(user_agent="elderly_care_gis_research")
    geocode = RateLimiter(geolocator.geocode, min_delay_seconds=delay_sec)

    lats, lons = [], []
    for addr in df[address_col]:
        try:
            loc = geocode(addr)
        except Exception:
            loc = None
        lats.append(loc.latitude if loc else np.nan)
        lons.append(loc.longitude if loc else np.nan)

    df = df.copy()
    df["lat"] = lats
    df["lon"] = lons
    df = df.dropna(subset=["lat", "lon"])

    geometry = [Point(xy) for xy in zip(df["lon"], df["lat"])]
    gdf = gpd.GeoDataFrame(df, geometry=geometry, crs=CRS_WGS84)
    return gdf.to_crs(CRS_TWD97)

# clinics_raw = fetch_nhi_medical_institutions(resource_id="A21030000I-D21004-009")
# medical_facilities = geocode_addresses(clinics_raw, address_col="地址")
# 註：地理編碼上千筆資料相當耗時（每筆需間隔 1 秒以上），建議先用小樣本測試，
#     正式跑全量時可考慮多執行緒（注意 Nominatim 使用條款禁止高併發）或改申請 TGOS。


### 3.4 無障礙設施與新北市開放資料通用介接方式

台北市與新北市個別提供開放資料平台，介接方式略有不同：
- **台北市**：多數資料集直接提供 CSV 下載網址（含經緯度），見 `data.gov.tw` 搜尋「台北市 無障礙」
- **新北市**（您提供的入口 `data.ntpc.gov.tw/openapi`）：所有資料集皆遵循統一 URL 格式，
  不需要另外申請金鑰


In [ ]:
def load_point_data_from_csv(csv_path_or_url, lon_col="lon", lat_col="lat", crs=CRS_WGS84):
    """通用函式：讀取含經緯度欄位的 CSV（本地路徑或網址皆可），轉為 GeoDataFrame。"""
    df = pd.read_csv(csv_path_or_url)
    df = df.dropna(subset=[lon_col, lat_col])
    geometry = [Point(xy) for xy in zip(df[lon_col], df[lat_col])]
    gdf = gpd.GeoDataFrame(df, geometry=geometry, crs=crs)
    return gdf.to_crs(CRS_TWD97)


def fetch_ntpc_dataset(dataset_oid, fmt="csv"):
    """新北市資料開放平臺通用介接函式。
    dataset_oid：於各資料集頁面網址最後一段取得（例如公園綠地、無障礙停車格等資料集）。
    使用方式：先至 https://data.ntpc.gov.tw 搜尋所需資料集（如「公園綠地」「無障礙」），
    複製網址列最後的資料代碼即為 dataset_oid。
    """
    url = f"https://data.ntpc.gov.tw/api/datasets/{dataset_oid}/{fmt}/file"
    df = pd.read_csv(url)
    return df

# 台北市無障礙友善店家（已含經緯度，資料集：data.gov.tw/dataset/147918）：
# accessible_tp = load_point_data_from_csv(
#     "<在 data.gov.tw/dataset/147918 頁面取得之 CSV 下載網址>",
#     lon_col="經度", lat_col="緯度",
# )

# 新北市無障礙 / 公園等資料（請先至 data.ntpc.gov.tw 找到對應 dataset_oid）：
# park_ntpc_raw = fetch_ntpc_dataset(dataset_oid="<複製自資料集網址>")
# 若該資料集含經緯度欄位，可直接用 load_point_data_from_csv() 轉換；
# 若僅含地址（部分新北市資料集如此），則需比照 3.3 節先做地理編碼。


## 4. 空間指標計算函式庫

以下函式為通用工具，套用於任何「村里代表點 × 設施點」的組合，
即可算出：**最近距離、路網距離/旅行時間、Buffer 內設施數量、服務範圍涵蓋率**。


In [ ]:
def nearest_distance(origin_gdf, target_gdf, id_col="village_code"):
    """用 BallTree 計算 origin（如村里中心點）到 target（如最近醫療機構）的
    歐氏最近距離（公尺）。origin_gdf / target_gdf 需為同一投影座標系（TWD97）。"""
    if len(target_gdf) == 0:
        result = origin_gdf[[id_col]].copy()
        result["nearest_dist_m"] = np.nan
        return result

    target_coords = np.array(list(zip(target_gdf.geometry.x, target_gdf.geometry.y)))
    origin_coords = np.array(list(zip(origin_gdf.geometry.x, origin_gdf.geometry.y)))

    tree = BallTree(target_coords, metric="euclidean")
    dist, idx = tree.query(origin_coords, k=1)

    result = origin_gdf[[id_col]].copy()
    result["nearest_dist_m"] = dist.flatten()
    return result


In [ ]:
def buffer_count(origin_gdf, target_gdf, radius_m, id_col="village_code"):
    """計算 origin 點半徑 radius_m 公尺內的 target 設施數量（可近性/密度指標）。"""
    origin_buffer = origin_gdf.copy()
    origin_buffer["geometry"] = origin_buffer.geometry.buffer(radius_m)

    if len(target_gdf) == 0:
        origin_gdf_out = origin_gdf[[id_col]].copy()
        origin_gdf_out[f"count_within_{radius_m}m"] = 0
        return origin_gdf_out

    joined = gpd.sjoin(target_gdf, origin_buffer[[id_col, "geometry"]], predicate="within")
    counts = joined.groupby(id_col).size().rename(f"count_within_{radius_m}m").reset_index()

    result = origin_gdf[[id_col]].merge(counts, on=id_col, how="left")
    result[f"count_within_{radius_m}m"] = result[f"count_within_{radius_m}m"].fillna(0)
    return result


In [ ]:
def polygon_buffer_coverage(village_gdf, facility_buffer_polys, id_col="village_code"):
    """計算「設施 buffer 聯集」與各村里面積的重疊比例（涵蓋率 0~1）。
    facility_buffer_polys：設施點做完 buffer 後的幾何聯集（unary_union 結果）。"""
    records = []
    for _, row in village_gdf.iterrows():
        village_geom = row.geometry
        inter_area = village_geom.intersection(facility_buffer_polys).area
        coverage_ratio = inter_area / village_geom.area if village_geom.area > 0 else np.nan
        records.append({id_col: row[id_col], "coverage_ratio": coverage_ratio})
    return pd.DataFrame(records)


In [ ]:
def network_distance_and_time(G, origin_points_gdf, target_points_gdf,
                               id_col="village_code", weight="length", speed_kmh=4.5):
    """路網最短距離（公尺）與估計旅行時間（分鐘）。
    G 需為 TWD97 或已投影座標之路網圖（osmnx graph 預設為 WGS84，
    建議先 ox.project_graph(G) 轉為公尺座標）。
    origin/target points 需為對應投影座標的 GeoDataFrame。
    """
    # 將節點座標對應到最近的路網節點
    origin_nodes = ox.distance.nearest_nodes(
        G, X=origin_points_gdf.geometry.x, Y=origin_points_gdf.geometry.y
    )
    target_nodes = ox.distance.nearest_nodes(
        G, X=target_points_gdf.geometry.x, Y=target_points_gdf.geometry.y
    )
    target_node_set = list(set(target_nodes))

    results = []
    for vid, o_node in zip(origin_points_gdf[id_col], origin_nodes):
        best_dist = np.inf
        for t_node in target_node_set:
            try:
                d = nx.shortest_path_length(G, o_node, t_node, weight=weight)
                if d < best_dist:
                    best_dist = d
            except nx.NetworkXNoPath:
                continue
        travel_time_min = (best_dist / 1000 / speed_kmh) * 60 if best_dist != np.inf else np.nan
        results.append({
            id_col: vid,
            "network_dist_m": best_dist if best_dist != np.inf else np.nan,
            "travel_time_min": travel_time_min,
        })
    return pd.DataFrame(results)

# 註：村里數 x 設施數量大時，上方逐一 shortest_path_length 效率較低，
# 資料量大時建議改用 nx.multi_source_dijkstra 一次以所有設施節點為起點做多源最短路徑，
# 或用 scipy.sparse.csgraph.dijkstra 向量化加速。


In [ ]:
def service_area_coverage(G, facility_points_gdf, village_gdf,
                          trip_time_min=15, speed_kmh=4.5, id_col="village_code"):
    """服務範圍（isochrone）涵蓋率：以每個設施為起點，在路網上找出
    trip_time_min 分鐘內可達的節點範圍，聯集後與村里界疊合算涵蓋率。"""
    cutoff_m = trip_time_min * (speed_kmh * 1000 / 60)

    facility_nodes = ox.distance.nearest_nodes(
        G, X=facility_points_gdf.geometry.x, Y=facility_points_gdf.geometry.y
    )

    reachable_points = []
    for node in facility_nodes:
        sub = nx.ego_graph(G, node, radius=cutoff_m, distance="length")
        for n, data in sub.nodes(data=True):
            reachable_points.append(Point(data["x"], data["y"]))

    if not reachable_points:
        return pd.DataFrame(columns=[id_col, f"service_coverage_{trip_time_min}min"])

    reachable_gdf = gpd.GeoDataFrame(geometry=reachable_points, crs=G.graph.get("crs", CRS_TWD97))
    # 用可達節點的凸包近似服務範圍面（實務上可改用 alphashape 做較貼合的凹包）
    service_polygon = reachable_gdf.unary_union.convex_hull.buffer(50)

    coverage = polygon_buffer_coverage(village_gdf, service_polygon, id_col=id_col)
    coverage = coverage.rename(columns={"coverage_ratio": f"service_coverage_{trip_time_min}min"})
    return coverage


## 5. 各指標計算流程（範例整合）

以下示範如何套用第 4 節的通用函式，計算各項指標。實際執行前，
請先完成第 1～3 節的資料下載（`village_gdf`, `G_walk`, `poi_dict`, 各政府開放資料等）。


In [ ]:
def compute_all_indicators(village_gdf, village_centroids, G_walk,
                            poi_dict, medical_gdf=None, ltc_gdf=None,
                            accessible_gdf=None, transit_gdf=None):
    """彙整所有村里層級指標，回傳單一寬表 DataFrame（以 village_code 為主鍵）。"""
    base = village_gdf[["village_code", "county", "district", "village", "village_area_km2"]].copy()

    # 1) 大眾運輸可近性：最近站點距離 + 500m 內站點數
    if transit_gdf is not None:
        d = nearest_distance(village_centroids, transit_gdf).rename(
            columns={"nearest_dist_m": "transit_nearest_dist_m"})
        c = buffer_count(village_centroids, transit_gdf, 500).rename(
            columns={"count_within_500m": "transit_count_500m"})
        base = base.merge(d, on="village_code").merge(c, on="village_code")

    # 2) 公園綠地可近性
    if "park" in poi_dict:
        d = nearest_distance(village_centroids, poi_dict["park"]).rename(
            columns={"nearest_dist_m": "park_nearest_dist_m"})
        c = buffer_count(village_centroids, poi_dict["park"], 500).rename(
            columns={"count_within_500m": "park_count_500m"})
        base = base.merge(d, on="village_code").merge(c, on="village_code")

    # 3) 日常生活機能可近性（超商、市場、藥局綜合）
    daily_life_pois = pd.concat(
        [poi_dict[k] for k in ["convenience_store", "supermarket", "market", "pharmacy"] if k in poi_dict],
        ignore_index=True,
    ) if poi_dict else gpd.GeoDataFrame()
    if len(daily_life_pois) > 0:
        daily_life_pois = gpd.GeoDataFrame(daily_life_pois, crs=CRS_TWD97)
        d = nearest_distance(village_centroids, daily_life_pois).rename(
            columns={"nearest_dist_m": "daily_life_nearest_dist_m"})
        c = buffer_count(village_centroids, daily_life_pois, 500).rename(
            columns={"count_within_500m": "daily_life_count_500m"})
        base = base.merge(d, on="village_code").merge(c, on="village_code")

    # 4) 高齡友善設施密度（社區關懷据點、日照中心等）
    if "community_center" in poi_dict:
        c = buffer_count(village_centroids, poi_dict["community_center"], 1000).rename(
            columns={"count_within_1000m": "elderly_facility_count_1000m"})
        base = base.merge(c, on="village_code")
        base["elderly_facility_density_per_km2"] = (
            base["elderly_facility_count_1000m"] / base["village_area_km2"]
        )

    # 5) 無障礙公共設施覆蓋率
    if accessible_gdf is not None:
        c = buffer_count(village_centroids, accessible_gdf, 500).rename(
            columns={"count_within_500m": "accessible_facility_count_500m"})
        base = base.merge(c, on="village_code")

    # 6) 醫療機構：最近距離 + 路網距離/旅行時間
    if medical_gdf is not None:
        d = nearest_distance(village_centroids, medical_gdf).rename(
            columns={"nearest_dist_m": "medical_nearest_dist_m"})
        base = base.merge(d, on="village_code")
        if G_walk is not None:
            nt = network_distance_and_time(G_walk, village_centroids, medical_gdf)
            nt = nt.rename(columns={
                "network_dist_m": "medical_network_dist_m",
                "travel_time_min": "medical_travel_time_min",
            })
            base = base.merge(nt, on="village_code")

    # 7) 長照機構：最近距離 + 服務範圍涵蓋率
    if ltc_gdf is not None:
        d = nearest_distance(village_centroids, ltc_gdf).rename(
            columns={"nearest_dist_m": "ltc_nearest_dist_m"})
        base = base.merge(d, on="village_code")
        if G_walk is not None:
            cov = service_area_coverage(G_walk, ltc_gdf, village_gdf, trip_time_min=15)
            base = base.merge(cov, on="village_code")

    return base

# all_indicators_village = compute_all_indicators(
#     village_gdf, village_centroids, G_walk, poi_dict,
#     medical_gdf=medical_facilities, ltc_gdf=ltc_facilities,
#     accessible_gdf=accessible_facilities, transit_gdf=bus_stops_tp,
# )
# all_indicators_village.head()


## 6. 依行政區彙整（村里 → 區）

村里層級指標可再依行政區加權平均或加總，產生「區級」摘要表，
兩種顆粒度都輸出，供後續模型依需求選用。


In [ ]:
def aggregate_to_district(village_indicators_df, weight_col="village_area_km2"):
    """將村里層級指標彙整為行政區層級（面積加權平均；計數類欄位改為加總）。"""
    df = village_indicators_df.copy()

    count_cols = [c for c in df.columns if "count" in c]
    dist_time_cols = [c for c in df.columns if ("dist" in c) or ("time" in c) or ("coverage" in c) or ("density" in c)]

    agg_dict = {}
    for c in count_cols:
        agg_dict[c] = "sum"
    for c in dist_time_cols:
        agg_dict[c] = "mean"  # 亦可改為面積加權平均，視研究需求調整

    district_df = df.groupby(["county", "district"]).agg(agg_dict).reset_index()
    district_df["village_count"] = df.groupby(["county", "district"])["village"].transform("count").drop_duplicates()
    return district_df

# district_indicators = aggregate_to_district(all_indicators_village)


## 7. 輸出 CSV

In [ ]:
def export_to_csv(village_df, district_df, output_dir=OUTPUT_DIR):
    village_path = os.path.join(output_dir, "gis_indicators_village_level.csv")
    district_path = os.path.join(output_dir, "gis_indicators_district_level.csv")

    village_df.to_csv(village_path, index=False, encoding="utf-8-sig")
    district_df.to_csv(district_path, index=False, encoding="utf-8-sig")

    print(f"已輸出：{village_path}")
    print(f"已輸出：{district_path}")

# export_to_csv(all_indicators_village, district_indicators)


## 8. 完整流程執行範例（Pipeline）

將以上步驟串成一個主流程，實際執行時請先確認：
1. 已下載村里界線檔案並放入 `data/raw/`
2. 已申請 TDX API 金鑰（或改用手動下載之公車/捷運站 CSV）
3. 已下載醫療、長照、無障礙設施資料 CSV 放入 `data/raw/`


In [ ]:
def run_pipeline(tdx_client_id=None, tdx_client_secret=None,
                  nhi_resource_ids=("A21030000I-D21004-009",),
                  accessible_csv_url=None):
    # tdx_client_id / tdx_client_secret：TDX 平台申請之金鑰（公車站牌、捷運車站用）
    # nhi_resource_ids：健保署醫事機構資料集 resource ID 清單（可含醫院、診所等多個）
    # accessible_csv_url：台北市無障礙友善店家等資料集之實際 CSV 下載網址
    print("Step 1：讀取行政區界線 ...")
    village_gdf = load_village_boundary()
    district_gdf = build_district_boundary(village_gdf)
    village_centroids = get_village_centroids(village_gdf)

    print("Step 2：下載道路網路 ...")
    G_walk = download_road_network(PLACE_NAMES, network_type="walk")
    G_walk = ox.project_graph(G_walk, to_crs=CRS_TWD97)

    print("Step 3：下載 OSM POI ...")
    poi_dict = {}
    for name, tags in OSM_TAGS.items():
        poi_dict[name] = download_osm_pois(PLACE_NAMES, tags)

    print("Step 4：讀取政府開放資料（長照 / 醫療 / 無障礙 / 公車 / 捷運） ...")
    ltc_gdf = fetch_ltc_abc_facilities()

    medical_frames = []
    for rid in nhi_resource_ids:
        raw = fetch_nhi_medical_institutions(resource_id=rid)
        medical_frames.append(geocode_addresses(raw, address_col="地址"))
    medical_gdf = gpd.GeoDataFrame(pd.concat(medical_frames, ignore_index=True), crs=CRS_TWD97)

    accessible_gdf = None
    if accessible_csv_url:
        accessible_gdf = load_point_data_from_csv(accessible_csv_url, lon_col="經度", lat_col="緯度")

    transit_gdf = None
    if tdx_client_id and tdx_client_secret:
        token = get_tdx_token(tdx_client_id, tdx_client_secret)
        bus_tp = fetch_tdx_bus_stops(token, city="Taipei")
        bus_ntp = fetch_tdx_bus_stops(token, city="NewTaipei")
        metro = fetch_tdx_metro_stations(token, rail_system="TRTC")
        transit_gdf = gpd.GeoDataFrame(pd.concat([bus_tp, bus_ntp, metro], ignore_index=True), crs=CRS_TWD97)

    print("Step 5：計算村里層級指標 ...")
    village_indicators = compute_all_indicators(
        village_gdf, village_centroids, G_walk, poi_dict,
        medical_gdf=medical_gdf, ltc_gdf=ltc_gdf,
        accessible_gdf=accessible_gdf, transit_gdf=transit_gdf,
    )

    print("Step 6：彙整行政區層級指標 ...")
    district_indicators = aggregate_to_district(village_indicators)

    print("Step 7：輸出 CSV ...")
    export_to_csv(village_indicators, district_indicators)

    return village_indicators, district_indicators

# village_result, district_result = run_pipeline(
#     tdx_client_id="YOUR_TDX_ID",
#     tdx_client_secret="YOUR_TDX_SECRET",
#     nhi_resource_ids=["A21030000I-D21004-009"],  # 診所；可再加醫院等 resource ID
#     accessible_csv_url="<data.gov.tw/dataset/147918 頁面之 CSV 下載網址>",
# )


## 8.1 快速示範：單一行政區（給接手組員的簡化版）

這個區塊獨立於上面的完整流程，**不需要先下載村里界線 shapefile**，
直接用 OSMnx 抓行政區邊界，範圍縮小到一個行政區（預設「大安區」），
執行時間約數分鐘，適合快速產出一份「能跑得出來、看得懂」的示範 CSV。

**只需要修改下方 config cell 中的三個地方：**
1. `TDX_CLIENT_ID`、`TDX_CLIENT_SECRET`：填入你申請到的 TDX **API** 金鑰
2. `DEMO_DISTRICT_EN`：行政區英文名稱（給 OSMnx 查詢用）
3. `DEMO_DISTRICT_ZH` / `DEMO_CITY_ZH`：行政區與縣市中文名稱（給政府資料篩選用）

改成新北市的區也可以，例如 `("Banqiao District, New Taipei, Taiwan", "板橋區", "新北市")`。


In [ ]:
# ============ 快速示範：設定區（請在這裡修改） ============
tdx_client_id = ""       # <-- 填入你的 TDX API 金鑰 Client ID
tdx_client_secret = ""   # <-- 填入你的 TDX API 金鑰 Client Secret

district_en = "Da'an District, Taipei, Taiwan"  # OSMnx 查詢用（英文地名）
district_zh = "大安區"                            # 政府資料篩選用（中文區名）
city_zh = "台北市"                                # 政府資料篩選用（中文縣市名）
# ============================================================


In [ ]:
# ============================================================
# 前置工具函式：請在 run_quick_demo() 之前執行
# ============================================================

import io
import numpy as np
import pandas as pd
import geopandas as gpd
import requests

from sklearn.neighbors import BallTree


def empty_point_gdf(crs=CRS_TWD97):
    """建立空的點位 GeoDataFrame。"""

    return gpd.GeoDataFrame(
        {"geometry": []},
        geometry="geometry",
        crs=crs
    )


# ============================================================
# 1. 衛福部長照 ABC 據點
# ============================================================

def fetch_ltc_abc_facilities(cities):
    """下載衛福部長照ABC據點，並轉成GeoDataFrame。"""

    csv_url = "https://ltcpap.mohw.gov.tw/publish/abc.csv"

    response = requests.get(
        csv_url,
        timeout=(15, 90),
        headers={
            "User-Agent": "Mozilla/5.0"
        }
    )

    response.raise_for_status()

    df = pd.read_csv(
        io.BytesIO(response.content),
        encoding="utf-8-sig"
    )

    # 清理欄位名稱
    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
        .str.replace("\ufeff", "", regex=False)
    )

    required_columns = [
        "特約縣市",
        "經度",
        "緯度"
    ]

    missing_columns = [
        col for col in required_columns
        if col not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"長照資料缺少欄位：{missing_columns}；"
            f"目前欄位：{list(df.columns)}"
        )

    # 將「台」和「臺」統一，避免台北市／臺北市無法匹配
    target_cities = {
        str(city).replace("台", "臺")
        for city in cities
    }

    city_normalized = (
        df["特約縣市"]
        .astype(str)
        .str.strip()
        .str.replace("台", "臺", regex=False)
    )

    df = df[
        city_normalized.isin(target_cities)
    ].copy()

    # 經緯度轉數字
    df["經度"] = pd.to_numeric(
        df["經度"],
        errors="coerce"
    )

    df["緯度"] = pd.to_numeric(
        df["緯度"],
        errors="coerce"
    )

    df = df.dropna(
        subset=["經度", "緯度"]
    )

    # 排除不合理座標
    df = df[
        df["經度"].between(119, 123)
        & df["緯度"].between(21, 26)
    ].copy()

    if df.empty:
        return empty_point_gdf(CRS_TWD97)

    ltc_gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(
            df["經度"],
            df["緯度"]
        ),
        crs=CRS_WGS84
    )

    return ltc_gdf.to_crs(CRS_TWD97)


# ============================================================
# 2. TDX Token
# ============================================================

def get_tdx_token(client_id, client_secret):
    """使用Client ID與Client Secret取得TDX存取權杖。"""

    token_url = (
        "https://tdx.transportdata.tw/auth/realms/"
        "TDXConnect/protocol/openid-connect/token"
    )

    response = requests.post(
        token_url,
        data={
            "grant_type": "client_credentials",
            "client_id": client_id,
            "client_secret": client_secret
        },
        timeout=30
    )

    response.raise_for_status()

    token_data = response.json()

    if "access_token" not in token_data:
        raise ValueError(
            f"TDX回傳資料沒有access_token：{token_data}"
        )

    return token_data["access_token"]


def extract_tdx_records(data):
    """相容TDX可能出現的不同回傳格式。"""

    if isinstance(data, list):
        return data

    if isinstance(data, dict):
        for key in ["value", "Data", "data"]:
            if isinstance(data.get(key), list):
                return data[key]

    return []


# ============================================================
# 3. TDX 公車站牌
# ============================================================

def fetch_tdx_bus_stops(token, city="Taipei"):
    """取得指定城市的TDX公車站牌。"""

    url = (
        "https://tdx.transportdata.tw/"
        f"api/basic/v2/Bus/Stop/City/{city}"
    )

    response = requests.get(
        url,
        headers={
            "Authorization": f"Bearer {token}",
            "Accept": "application/json"
        },
        params={
            "$format": "JSON"
        },
        timeout=90
    )

    response.raise_for_status()

    records = extract_tdx_records(
        response.json()
    )

    rows = []

    for item in records:
        position = item.get("StopPosition") or {}

        longitude = position.get("PositionLon")
        latitude = position.get("PositionLat")

        if longitude is None or latitude is None:
            continue

        stop_name = item.get("StopName") or {}

        rows.append({
            "stop_uid": item.get("StopUID"),
            "stop_id": item.get("StopID"),
            "stop_name": stop_name.get("Zh_tw"),
            "longitude": longitude,
            "latitude": latitude
        })

    if not rows:
        return empty_point_gdf(CRS_TWD97)

    df = pd.DataFrame(rows)

    bus_gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(
            df["longitude"],
            df["latitude"]
        ),
        crs=CRS_WGS84
    )

    return bus_gdf.to_crs(CRS_TWD97)


# ============================================================
# 4. TDX 捷運車站
# ============================================================

def fetch_tdx_metro_stations(token, rail_system="TRTC"):
    """取得指定捷運系統的車站資料。"""

    url = (
        "https://tdx.transportdata.tw/"
        f"api/basic/v2/Rail/Metro/Station/{rail_system}"
    )

    response = requests.get(
        url,
        headers={
            "Authorization": f"Bearer {token}",
            "Accept": "application/json"
        },
        params={
            "$format": "JSON"
        },
        timeout=90
    )

    response.raise_for_status()

    records = extract_tdx_records(
        response.json()
    )

    rows = []

    for item in records:
        position = item.get("StationPosition") or {}

        longitude = position.get("PositionLon")
        latitude = position.get("PositionLat")

        if longitude is None or latitude is None:
            continue

        station_name = item.get("StationName") or {}

        rows.append({
            "station_uid": item.get("StationUID"),
            "station_id": item.get("StationID"),
            "station_name": station_name.get("Zh_tw"),
            "longitude": longitude,
            "latitude": latitude
        })

    if not rows:
        return empty_point_gdf(CRS_TWD97)

    df = pd.DataFrame(rows)

    metro_gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(
            df["longitude"],
            df["latitude"]
        ),
        crs=CRS_WGS84
    )

    return metro_gdf.to_crs(CRS_TWD97)


# ============================================================
# 5. 最近設施距離
# ============================================================

def nearest_distance(
    source_gdf,
    facility_gdf,
    id_col="village_code"
):
    """計算每個來源點到最近設施的直線距離（公尺）。"""

    source = source_gdf.copy()
    facilities = facility_gdf.copy()

    if source.crs != CRS_TWD97:
        source = source.to_crs(CRS_TWD97)

    if facilities.crs != CRS_TWD97:
        facilities = facilities.to_crs(CRS_TWD97)

    source_valid = (
        source.geometry.notna()
        & ~source.geometry.is_empty
    )

    facility_valid = (
        facilities.geometry.notna()
        & ~facilities.geometry.is_empty
    )

    source = source[source_valid].copy()
    facilities = facilities[facility_valid].copy()

    if source.empty:
        return pd.DataFrame(
            columns=[id_col, "nearest_dist_m"]
        )

    if facilities.empty:
        return pd.DataFrame({
            id_col: source[id_col].values,
            "nearest_dist_m": [np.nan] * len(source)
        })

    source_coordinates = np.column_stack([
        source.geometry.x,
        source.geometry.y
    ])

    facility_coordinates = np.column_stack([
        facilities.geometry.x,
        facilities.geometry.y
    ])

    tree = BallTree(
        facility_coordinates,
        metric="euclidean"
    )

    distances, _ = tree.query(
        source_coordinates,
        k=1
    )

    return pd.DataFrame({
        id_col: source[id_col].values,
        "nearest_dist_m": distances[:, 0]
    })


# ============================================================
# 6. Buffer範圍內設施數量
# ============================================================

def buffer_count(
    source_gdf,
    facility_gdf,
    radius_m,
    id_col="village_code"
):
    """計算每個來源點指定公尺內的設施數量。"""

    source = source_gdf.copy()
    facilities = facility_gdf.copy()

    if source.crs != CRS_TWD97:
        source = source.to_crs(CRS_TWD97)

    if facilities.crs != CRS_TWD97:
        facilities = facilities.to_crs(CRS_TWD97)

    facilities = facilities[
        facilities.geometry.notna()
        & ~facilities.geometry.is_empty
    ].copy()

    counts = []

    for geometry in source.geometry:
        if geometry is None or geometry.is_empty:
            counts.append(np.nan)
            continue

        if facilities.empty:
            counts.append(0)
            continue

        inside_count = (
            facilities.geometry.distance(geometry)
            <= radius_m
        ).sum()

        counts.append(int(inside_count))

    column_name = f"count_within_{int(radius_m)}m"

    return pd.DataFrame({
        id_col: source[id_col].values,
        column_name: counts
    })


print("前置工具函式載入完成")

前置工具函式載入完成


In [ ]:
def run_quick_demo(district_en, district_zh, city_zh,
                    tdx_client_id, tdx_client_secret):
    """單一行政區快速示範：抓資料 → 算基礎指標 → 輸出 CSV。
    僅計算「Buffer 內設施數量」與「最近距離」，不含耗時的路網最短路徑分析，
    目的是讓沒有事先準備任何檔案的人也能在數分鐘內跑出一份成果。"""

    print(f"[1/6] 取得「{district_en}」行政區邊界（OSMnx，不需手動下載檔案）...")
    district_gdf = ox.geocode_to_gdf(district_en)
    district_gdf = district_gdf.to_crs(CRS_TWD97)
    district_polygon = district_gdf.geometry.iloc[0]
    district_centroid = gpd.GeoDataFrame(
        {"district": [district_zh]}, geometry=[district_polygon.centroid], crs=CRS_TWD97
    )

    print("[2/6] 下載 OSM 生活機能 POI（超商、公園、藥局、社區關懷据點）...")
    demo_tags = {
        "convenience_store": {"shop": "convenience"},
        "park": {"leisure": "park"},
        "pharmacy": {"amenity": "pharmacy"},
        "community_center": {"amenity": "community_centre"},
    }
    poi_dict = {}
    for name, tags in demo_tags.items():
        try:
            gdf = ox.features_from_place(district_en, tags)
            gdf["geometry"] = gdf.geometry.apply(
                lambda geom: geom.centroid if geom.geom_type != "Point" else geom
            )
            poi_dict[name] = gpd.GeoDataFrame(gdf, crs=CRS_WGS84).to_crs(CRS_TWD97)
        except Exception as e:
            print(f"  {name} 下載失敗：{e}")
            poi_dict[name] = gpd.GeoDataFrame(geometry=[], crs=CRS_TWD97)

        print("[3/6] 取得長照ABC據點（衛福部，已含經緯度）...")
        ltc_download_ok = False

    ltc_in_district = gpd.GeoDataFrame(
        {"geometry": []},
        geometry="geometry",
        crs=CRS_TWD97
    )

    try:
        if "fetch_ltc_abc_facilities" not in globals():
            raise NameError(
                "尚未定義 fetch_ltc_abc_facilities()"
            )

        ltc_gdf = fetch_ltc_abc_facilities(
            cities=(city_zh,)
        )

        if ltc_gdf.crs != district_gdf.crs:
            ltc_gdf = ltc_gdf.to_crs(CRS_TWD97)

        ltc_in_district = ltc_gdf[
            ltc_gdf.geometry.within(district_polygon)
        ].copy()

        ltc_download_ok = True

        print(
            f"  長照資料取得成功："
            f"{district_zh}共有 {len(ltc_in_district)} 個據點"
        )

    except Exception as e:
        print(f"  長照資料暫時無法取得：{e}")
        print("  本次先略過長照資料，繼續執行後續分析。")

        print("[4/6] 取得公車站牌與捷運車站（TDX API）...")
        transit_gdf = gpd.GeoDataFrame(geometry=[], crs=CRS_TWD97)
        if tdx_client_id and tdx_client_secret:
            try:
                token = get_tdx_token(tdx_client_id, tdx_client_secret)
                city_en = "Taipei" if city_zh == "台北市" else "NewTaipei"
                bus = fetch_tdx_bus_stops(token, city=city_en)
                bus_in_district = bus[bus.geometry.within(district_polygon)]
                frames = [bus_in_district]
                if city_zh == "台北市":
                    metro = fetch_tdx_metro_stations(token, rail_system="TRTC")
                    frames.append(metro[metro.geometry.within(district_polygon)])
                transit_gdf = gpd.GeoDataFrame(pd.concat(frames, ignore_index=True), crs=CRS_TWD97)
            except Exception as e:
                print(f"  TDX 資料下載失敗（請確認金鑰是否正確）：{e}")
        else:
            print("  未提供 TDX 金鑰，略過公車／捷運資料")

        print("[5/6] 計算指標（以行政區中心點為代表點）...")
        records = {"county": city_zh, "district": district_zh}

        for name, gdf in poi_dict.items():
            d = nearest_distance(district_centroid.assign(village_code=district_zh), gdf, id_col="village_code")
            c = buffer_count(district_centroid.assign(village_code=district_zh), gdf, 500, id_col="village_code")
            records[f"{name}_nearest_dist_m"] = d["nearest_dist_m"].iloc[0]
            records[f"{name}_count_500m"] = c["count_within_500m"].iloc[0]

        records["ltc_facility_count_in_district"] = (
        len(ltc_in_district)
        if ltc_download_ok
        else pd.NA
    )
        records["transit_stop_count_in_district"] = len(transit_gdf)

        print("[6/6] 輸出 CSV ...")
        result_df = pd.DataFrame([records])
        out_path = os.path.join(OUTPUT_DIR, f"demo_{district_zh}_indicators.csv")
        result_df.to_csv(out_path, index=False, encoding="utf-8-sig")
        print(f"完成！已輸出：{out_path}")

        return result_df

demo_result = run_quick_demo(
    district_en=DEMO_DISTRICT_EN,
    district_zh=DEMO_DISTRICT_ZH,
    city_zh=DEMO_CITY_ZH,
    tdx_client_id=TDX_CLIENT_ID,
    tdx_client_secret=TDX_CLIENT_SECRET,
)
demo_result


[1/6] 取得「Da'an District, Taipei, Taiwan」行政區邊界（OSMnx，不需手動下載檔案）...
[2/6] 下載 OSM 生活機能 POI（超商、公園、藥局、社區關懷据點）...
[3/6] 取得長照ABC據點（衛福部，已含經緯度）...
[3/6] 取得長照ABC據點（衛福部，已含經緯度）...
[3/6] 取得長照ABC據點（衛福部，已含經緯度）...
[3/6] 取得長照ABC據點（衛福部，已含經緯度）...
  長照資料暫時無法取得：HTTPSConnectionPool(host='ltcpap.mohw.gov.tw', port=443): Max retries exceeded with url: /publish/abc.csv (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x7dfc91ac0a70>, 'Connection to ltcpap.mohw.gov.tw timed out. (connect timeout=15)'))
  本次先略過長照資料，繼續執行後續分析。
[4/6] 取得公車站牌與捷運車站（TDX API）...
[5/6] 計算指標（以行政區中心點為代表點）...
[6/6] 輸出 CSV ...
完成！已輸出：outputs/demo_大安區_indicators.csv


,county,district,convenience_store_nearest_dist_m,convenience_store_count_500m,park_nearest_dist_m,park_count_500m,pharmacy_nearest_dist_m,pharmacy_count_500m,community_center_nearest_dist_m,community_center_count_500m,ltc_facility_count_in_district,transit_stop_count_in_district
0,台北市,大安區,69.102417,22,126.128692,14,172.156765,13,113.349745,3,<NA>,1975


## 9. 資料可重複更新性與自動化建議

- 將本 Notebook 轉為 `.py` script（`jupyter nbconvert --to script`），
  搭配 cron / GitHub Actions 排程定期重跑，確保資料定期更新
- OSM 資料具高更新頻率但品質不一，建議加入資料檢核（如座標是否落在研究範圍內、
  是否有缺漏欄位）後再納入分析
- 政府開放資料版本常有變動（欄位名稱、檔案格式），建議將欄位對照表
  （如 `rename_map`）獨立成 config 檔，方便日後資料源更新時快速調整
- 大量村里 × 大量設施點的路網最短路徑計算量大，若效能不足，建議：
  - 改用 `networkx.multi_source_dijkstra`（多起點一次計算）
  - 或改用 `pandana` 套件做大規模路網可近性分析（比 networkx 快非常多）
- 建議將 `village_code`（政府標準村里代碼）作為所有表格的共同主鍵，
  避免因中文地名不一致（如「一」vs「1」）造成合併錯誤
